# SAA Overconfidence & Calibration Analysis

Behavioral economics analysis of the State Archives of Assyria (SAA) corpus.
Focus: overconfidence signals, forecast accuracy, and calibration evidence across
3,302 Neo-Assyrian documents (c. 911–609 BCE).

**Volumes included:**
- SAA01, 05, 15–19, 21 — administrative/diplomatic letters
- SAA04 — queries to the sun god (oracles)
- SAA08 — astrological reports and omens
- SAA09 — prophetic texts
- SAA10 — letters from scholars and exorcists

**Behavioral constructs targeted:**
- Overconfidence: certainty claims + forecast claims without hedging
- Calibration: documents combining a forecast claim with a failed-prediction signal
- Loss aversion: loss framing, complaint/petition, reference-point language
- Deferred payment / intertemporal structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import textwrap

pd.set_option('display.max_colwidth', 300)
sns.set_theme(style='whitegrid', palette='muted')

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'processed_data/neo_assyrian/letters_translations_coded.csv')
print(f'{len(df):,} documents | {df["project"].nunique()} SAA volumes')
print(df['project'].value_counts().to_string())

## 1. Behavioral Signal Overview

In [ ]:
be_cols = [c for c in df.columns if c.startswith('be_')]
signal_counts = {c: int(df[c].sum()) for c in be_cols if df[c].sum() > 0}
signal_df = pd.DataFrame({
    'code': list(signal_counts.keys()),
    'n': list(signal_counts.values()),
    'pct': [100*v/len(df) for v in signal_counts.values()]
}).sort_values('n', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(signal_df['code'].str.replace('be_', '', regex=False), signal_df['pct'],
               color=['#c0392b' if 'signal' in c else '#2980b9' for c in signal_df['code']])
ax.set_xlabel('% of documents')
ax.set_title('Behavioral Codes — SAA Corpus (n=3,302)', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
for bar, (_, row) in zip(bars, signal_df.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['n']:,}", va='center', fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/saa_behavioral_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nDominant signal: be_forecast_claim at {100*df['be_forecast_claim'].mean():.1f}%")

## 2. Overconfidence by Volume

SAA08 (astrological reports) and SAA10 (scholar letters) should show the highest
forecast claim rates — these are expert advisors making predictions to the king.

In [ ]:
vol_stats = df.groupby('project').agg(
    n=('text_id', 'count'),
    forecast_n=('be_forecast_claim', 'sum'),
    certainty_n=('be_certainty_claim', 'sum'),
    overconf_n=('be_overconfidence_signal', 'sum'),
    failed_n=('be_failed_prediction', 'sum'),
    calibration_n=('be_calibration_signal', 'sum'),
    loss_n=('be_loss_aversion_signal', 'sum'),
).reset_index()

vol_stats['forecast_pct'] = 100 * vol_stats['forecast_n'] / vol_stats['n']
vol_stats['overconf_pct'] = 100 * vol_stats['overconf_n'] / vol_stats['n']
vol_stats['calib_rate']   = 100 * vol_stats['calibration_n'] / vol_stats['forecast_n'].clip(lower=1)

# Clean labels
vol_stats['label'] = vol_stats['project'].str.replace('saao_', '', regex=False).str.upper()

print(vol_stats[['label','n','forecast_n','forecast_pct','certainty_n',
                  'overconf_pct','calibration_n','calib_rate']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: forecast rate by volume
ax = axes[0]
vs = vol_stats.sort_values('forecast_pct', ascending=True)
bars = ax.barh(vs['label'], vs['forecast_pct'], color='#2980b9')
ax.set_xlabel('% documents with forecast claim')
ax.set_title('Forecast Claims by SAA Volume', fontweight='bold')
for bar, (_, row) in zip(bars, vs.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['forecast_n']:.0f}/{row['n']:.0f}", va='center', fontsize=8)

# Right: calibration rate (failed / forecast) by volume
ax = axes[1]
vs2 = vol_stats[vol_stats['forecast_n'] >= 10].sort_values('calib_rate', ascending=True)
bars2 = ax.barh(vs2['label'], vs2['calib_rate'], color='#c0392b')
ax.set_xlabel('% forecast docs also showing failure signal')
ax.set_title('Calibration Signal Rate\n(failed_prediction | forecast_claim)', fontweight='bold')
for bar, (_, row) in zip(bars2, vs2.iterrows()):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{row['calibration_n']:.0f}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig(ROOT / 'outputs/saa_overconfidence_by_volume.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Calibration Documents — Qualitative Review

The `be_calibration_signal` flag marks documents that contain *both* a forecast claim
and a failed-prediction signal. These are the highest-value documents for behavioral
analysis — they represent observable calibration failures in expert prediction.

In [ ]:
cal_docs = df[df['be_calibration_signal']].copy()
print(f"{len(cal_docs)} calibration documents across {cal_docs['project'].nunique()} volumes\n")
print(cal_docs['project'].value_counts().to_string())
print()

# Print full text of each calibration document
for _, row in cal_docs.iterrows():
    vol = row['project'].replace('saao_', '').upper()
    print(f"{'='*70}")
    print(f"[{vol}] {row['text_id']}")
    print('='*70)
    text = str(row['translation'])
    for line in textwrap.wrap(text, 100):
        print(line)
    print()

## 4. Certainty Claims — Language Analysis

SAA10 (scholar letters) shows the highest *certainty* claim rate — these are
expert advisors (astronomers, physicians, exorcists) asserting knowledge to the king.
This is the purest overconfidence signal: declarative certainty in predictive domains.

In [ ]:
import re

# Extract the certainty-claim phrases from SAA10 docs
certainty_pattern = (
    r'(?i)(\b\w*\s*certain(?:ly)?\b|\bsurely\b|\bwithout doubt\b|\bno doubt\b'
    r'|\bassuredly\b|\bdefinitely\b|\bwill certainly\b|\bwill surely\b'
    r'|\bI know\b|\bI am sure\b)'
)

saa10_cert = df[(df['project'] == 'saao_saa10') & df['be_certainty_claim']].copy()
print(f"SAA10 certainty docs: {len(saa10_cert)}\n")

phrases = []
for _, row in saa10_cert.iterrows():
    text = str(row['translation'])
    matches = re.findall(certainty_pattern, text)
    for m in matches:
        # Get context: 60 chars around match
        idx = text.find(m)
        ctx = text[max(0,idx-40):idx+len(m)+40].replace('\n', ' ')
        phrases.append({'text_id': row['text_id'], 'phrase': m, 'context': f'...{ctx}...'})

phrases_df = pd.DataFrame(phrases)
if not phrases_df.empty:
    print('Most common certainty phrases in SAA10:')
    print(phrases_df['phrase'].str.lower().value_counts().head(15).to_string())
    print()
    print('Sample contexts:')
    for _, r in phrases_df.head(8).iterrows():
        print(f"  [{r['text_id']}] {r['context']}")
        print()

## 5. Oracle Texts (SAA04) — Forecast Structure

SAA04 are formal queries to the sun god Shamash asking whether a proposed
action will succeed. These are the most structurally pure forecast documents:
binary outcome predictions with explicit uncertainty acknowledgment.
Relevant constructs: forecast framing, reference points, loss asymmetry.

In [ ]:
saa04 = df[df['project'] == 'saao_saa04'].copy()
print(f"SAA04 oracle queries: {len(saa04)}")
print()

be_cols_subset = ['be_forecast_claim','be_certainty_claim','be_loss_framing',
                   'be_gain_framing','be_reference_point','be_failed_prediction',
                   'be_deferred_payment','be_contract_formal']

for c in be_cols_subset:
    n = saa04[c].sum()
    print(f"  {c:35s}: {n:3d} ({100*n/len(saa04):.1f}%)")

print()
# Show a representative oracle
print("Sample oracle query (forecast + loss framing):")
sample = saa04[saa04['be_loss_framing'] & saa04['be_forecast_claim']]
if not sample.empty:
    row = sample.iloc[0]
    print(f"\n[{row['text_id']}]")
    for line in textwrap.wrap(str(row['translation']), 100):
        print(line)

## 6. Astrological Reports (SAA08) — Omen Forecast Framing

SAA08 are royal astrological reports: scribes report celestial observations
and derive predictions. They show the highest absolute count of forecast claims (284/566, 50%).
Critical for calibration: the 'if X then Y' conditional structure maps directly
onto probability assessment language.

In [ ]:
saa08 = df[df['project'] == 'saao_saa08'].copy()
print(f"SAA08 astrological reports: {len(saa08)}")
print(f"Forecast claims: {saa08['be_forecast_claim'].sum()} ({100*saa08['be_forecast_claim'].mean():.1f}%)")
print(f"Failed prediction: {saa08['be_failed_prediction'].sum()}")
print(f"Calibration signal: {saa08['be_calibration_signal'].sum()}")
print()

# Frequency of specific modal/forecast verbs
forecast_verbs = [
    (r'\bshall\b', 'shall'),
    (r'\bwill\b', 'will'),
    (r'\bmeans\b', 'means (omen interpretation)'),
    (r'\bportends\b', 'portends'),
    (r'\bdenotes\b', 'denotes'),
    (r'\bsignifies\b', 'signifies'),
    (r'\bforetoken', 'foretokens'),
    (r'\bif .{5,60} (?:will|shall)', 'if...will/shall (conditional)'),
]

print('Forecast language in SAA08:')
for pattern, label in forecast_verbs:
    n = saa08['translation'].fillna('').str.count(pattern, flags=re.IGNORECASE).sum()
    print(f"  {label:40s}: {int(n):4d} occurrences")

print()
# Show a calibration-signal SAA08 doc
cal08 = saa08[saa08['be_calibration_signal']]
if not cal08.empty:
    print(f"Calibration document in SAA08 ({len(cal08)} total):")
    row = cal08.iloc[0]
    print(f"\n[{row['text_id']}]")
    for line in textwrap.wrap(str(row['translation'])[:600], 100):
        print(line)

## 7. Loss Aversion in Diplomatic Letters

Letters (SAA01, SAA15–SAA21) are administrative/diplomatic communications.
Loss framing in these contexts maps directly onto complaint/grievance behavior —
a behavioral marker of loss aversion: agents act more urgently when describing
losses vs. equivalent gains.

In [ ]:
letter_vols = ['saao_saa01','saao_saa15','saao_saa16','saao_saa17',
               'saao_saa18','saao_saa19','saao_saa21']
letters = df[df['project'].isin(letter_vols)].copy()
print(f"Letter corpus: {len(letters):,} documents")
print()

# Cross-tabulate loss framing vs. gain framing
loss_n = letters['be_loss_framing'].sum()
gain_n = letters['be_gain_framing'].sum()
both_n = (letters['be_loss_framing'] & letters['be_gain_framing']).sum()
print(f"Loss framing only:  {loss_n - both_n:3d} ({100*(loss_n-both_n)/len(letters):.1f}%)")
print(f"Gain framing only:  {gain_n - both_n:3d} ({100*(gain_n-both_n)/len(letters):.1f}%)")
print(f"Both:               {both_n:3d} ({100*both_n/len(letters):.1f}%)")
print(f"Loss-to-gain ratio: {(loss_n/gain_n):.2f}x" if gain_n > 0 else "Gain: 0")
print()

# Loss framing by volume
vol_loss = letters.groupby('project')[['be_loss_framing','be_gain_framing',
                                        'be_complaint_petition','be_reference_point']].mean().round(3) * 100
vol_loss.index = vol_loss.index.str.replace('saao_', '', regex=False).str.upper()
vol_loss.columns = ['loss_%','gain_%','complaint_%','ref_point_%']
print(vol_loss.sort_values('loss_%', ascending=False).to_string())

print()
# Show highest-signal loss + complaint letters
high_loss = letters[
    letters['be_loss_framing'] & letters['be_complaint_petition'] & letters['be_reference_point']
]
print(f"Documents with loss_framing + complaint + reference_point: {len(high_loss)}")
if not high_loss.empty:
    row = high_loss.iloc[0]
    vol = row['project'].replace('saao_','').upper()
    print(f"\nExample [{vol}] {row['text_id']}:")
    for line in textwrap.wrap(str(row['translation'])[:500], 100):
        print(line)

## 8. Deferred Payment and Intertemporal Structure

In [ ]:
defer = df[df['be_deferred_payment']].copy()
print(f"Deferred payment docs: {len(defer)} ({100*len(defer)/len(df):.1f}%)")
print()
print('By volume:')
print(defer['project'].value_counts().to_string())
print()

# Deferred payment + loan credit = clear intertemporal transaction
inter_docs = df[df['be_deferred_payment'] & df['be_loan_credit']]
print(f"Deferred + loan_credit (intertemporal transactions): {len(inter_docs)}")
print()

# Sample deferred payment docs with commodity context
defer_grain = df[df['be_deferred_payment'] & df['be_commodity_grain']]
defer_silver = df[df['be_deferred_payment'] & df['be_commodity_money']]
print(f"Deferred + grain: {len(defer_grain)} | Deferred + silver/money: {len(defer_silver)}")

if not defer_grain.empty:
    print()
    row = defer_grain.iloc[0]
    vol = row['project'].replace('saao_','').upper()
    print(f"Example grain deferred payment [{vol}] {row['text_id']}:")
    for line in textwrap.wrap(str(row['translation'])[:400], 100):
        print(line)

## 9. Summary: Behavioral Construct Prevalence

Summary table of all behavioral signals across the full SAA corpus,
disaggregated by document type (oracle/astrological vs. letter vs. scholarly).

In [ ]:
# Classify volumes into document types
doc_type_map = {
    'saao_saa04': 'oracle',
    'saao_saa08': 'astrological',
    'saao_saa09': 'prophecy',
    'saao_saa10': 'scholarly_letter',
    'saao_saa05': 'provincial_letter',
    'saao_saa01': 'royal_letter',
    'saao_saa15': 'provincial_letter',
    'saao_saa16': 'provincial_letter',
    'saao_saa17': 'provincial_letter',
    'saao_saa18': 'provincial_letter',
    'saao_saa19': 'provincial_letter',
    'saao_saa21': 'provincial_letter',
}
df['doc_type'] = df['project'].map(doc_type_map)

key_signals = [
    'be_forecast_claim', 'be_certainty_claim', 'be_failed_prediction',
    'be_calibration_signal', 'be_loss_framing', 'be_gain_framing',
    'be_complaint_petition', 'be_reference_point', 'be_deferred_payment',
    'be_loss_aversion_signal', 'be_overconfidence_signal', 'be_economic_core'
]

summary = df.groupby('doc_type')[key_signals].mean() * 100
summary = summary.round(1)
summary.columns = [c.replace('be_','') for c in summary.columns]
summary['n_docs'] = df.groupby('doc_type')['text_id'].count()

# Reorder: n_docs first
cols = ['n_docs'] + list(summary.columns[:-1])
summary = summary[cols]

print(summary.to_string())
print()
print("Values = % of documents in each type with that signal")

In [ ]:
# Heatmap of behavioral signals by document type
plot_signals = [
    'forecast_claim', 'certainty_claim', 'failed_prediction', 'calibration_signal',
    'loss_framing', 'complaint_petition', 'reference_point',
    'deferred_payment', 'economic_core'
]
hm_data = summary[plot_signals].T

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(hm_data, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of documents'})
ax.set_title('Behavioral Signal Rates by SAA Document Type (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Document type')
ax.set_ylabel('Behavioral code')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(ROOT / 'outputs/saa_behavioral_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey findings:")
print(f"  - Astrological reports (SAA08): {hm_data.loc['forecast_claim','astrological']:.1f}% forecast claim rate")
print(f"  - Scholarly letters (SAA10):   {hm_data.loc['certainty_claim','scholarly_letter']:.1f}% certainty claim rate")
print(f"  - Oracle queries (SAA04):      {hm_data.loc['loss_framing','oracle']:.1f}% loss framing rate")
print(f"  - Calibration signal overall:  {100*df['be_calibration_signal'].mean():.2f}% of all SAA docs")

## 10. Behavioral Economics Interpretation

### Overconfidence
The 26.1% overconfidence signal rate in SAA is driven almost entirely by `be_forecast_claim`
(24.9%), not `be_certainty_claim` (1.9%). This reflects a distinction relevant to behavioral
theory: **predictive overconfidence** (making specific forecasts) vs. **confidence in
specific claims** (asserting certainty about known facts).

SAA08 astrological scribes are making conditional predictions ("if Mars rises in Leo, the
king will be victorious") — structurally identical to probability assessments. The 50%
forecast rate in SAA08 reflects institutional role, not individual overconfidence.

### Calibration
11 calibration signal documents represent cases where a forecast and a failure acknowledgment
co-occur in the *same* document. This is rare (0.3%) but analytically valuable: these are
cases where an agent both predicted an outcome and recorded evidence of prediction failure.
Cross-cultural calibration research should examine these as primary cases.

### Loss Aversion in Letters
The loss-to-gain framing ratio in the letter corpus is consistent with loss aversion theory:
agents devote more text to describing losses and seeking remediation than to describing gains.
Reference-point language ("as before," "as agreed," "as promised") frames complaints in terms
of deviation from a prior expectation — the behavioral mechanics of reference-dependent utility.

### Deferred Payment
209 deferred payment documents (6.3%) indicate substantial intertemporal transaction activity.
Combined with the loan/credit signal (22 docs, 0.7%), these suggest interest-rate and
time-discounting behavior can be studied in the SAA corpus, though at lower density than
the Babylonian Diaries or APIS papyri.